In [ ]:
workspace_id = ""
destination_lakehouse_id = ""
destination_lakehouse_path = ""
deployment_environment = ""

In [ ]:
from pyspark.sql.functions import input_file_name, col, count, countDistinct, coalesce, lit
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, LongType
from concurrent.futures import ThreadPoolExecutor
import unittest
from pyspark.sql import SparkSession
import io
import logging
import sempy.fabric as fabric
import xmlrunner
from delta import DeltaTable
from typing import Optional, List, Tuple, Dict
from pyspark.sql.utils import AnalysisException
from notebookutils import mssparkutils

# uncomment as needed picked these table names from adapter.json
EXPECTED_SILVER_TABLE_COUNTS = [
    "Patient",
    "Encounter",
    "Procedure",
    "ExplanationOfBenefit",
    "Observation",
    "CarePlan",
    # "SocialDeterminant",
    # "UnitOfMeasure",
    # "SocialDeterminantSubCategory",
    # "SocialDeterminantCategory",
    # "SocialDeterminantDataSetMetadata",
    # "ZipToFipsMapping"
]

# uncomment as needed picked these table names from dbtargetschemaconfig.json
EXPECTED_GOLD_TABLE_COUNTS = [
    "person",
    "death",
    "location",
    "visit_occurrence",
    "condition_occurrence",
    "procedure_occurrence",
    # "measurement",
    # "care_plan",
    # "care_plan_activities",
    # "care_plan_addresses",
    # "care_plan_goal",
    # "cost",
    # "social_determinant",
    # "sdoh_unitofmeasure",
    # "sdoh_category",
    # "sdoh_datasetmetadata",
    # "sdoh_location",
    # "ZipToFipsMapping",
    # "sdoh_fips",
    # "patient_location_zip"
]

class CMADataValidationTests(unittest.TestCase):

    def __init__(self, methodName='runTest', spark=None, workspace_id = None, bronze_lakehouse_id = None, databases = []):
        super().__init__(methodName)
        logging.basicConfig()
        self.logger = logging.getLogger("LOG")
        self.spark = spark
        self.workspace_id = workspace_id
        self.bronze_lakehouse_id = bronze_lakehouse_id
        self.databases = databases

    # add more expected_resources if needed to validate 
    def test_verify_resourcetypes(self):
        expected_resource_types = ["CarePlan"]
        df = self.spark.sql("SELECT DISTINCT resourceType FROM healthcare1_msft_bronze.ClinicalFhir")
        result = [row['resourceType'] for row in df.collect()] 

        # Verify that all expected resource types are present in the result
        for expected_resource_type in expected_resource_types:
            self.assertIn(expected_resource_type, result)

    def test_silver_table_hydration(self):
        for table in EXPECTED_SILVER_TABLE_COUNTS:
            query = f"SELECT COUNT(*) as count FROM healthcare1_msft_silver.{table}"
            df = self.spark.sql(query)
            count = df.collect()[0]['count']
            self.assertGreater(count, 0, f"Table {table} has no data. Count is {count}.")
    
    def test_gold_table_hydration(self):
        for table in EXPECTED_GOLD_TABLE_COUNTS:
            query = f"SELECT COUNT(*) as count FROM healthcare1_msft_gold_cma.{table}"
            df = self.spark.sql(query)
            count = df.collect()[0]['count']
            self.assertGreater(count, 0, f"Table {table} has no data. Count is {count}.")


def run_tests_and_write_output(spark):
    # Load and run tests
    loader = unittest.TestLoader()
    suite = loader.loadTestsFromTestCase(CMADataValidationTests)

    # Inject parameters / context to the tests
    for test in suite:
        test.spark = spark
        test.workspace_id = workspace_id

    # Write XML test report to stream, decode after completion
    write_stream = io.BytesIO()
    xmlrunner.XMLTestRunner(output=write_stream).run(suite)
    xml_output = write_stream.getvalue().decode('utf-8')

    # Write report to lakehouse
    mssparkutils.fs.put(f"abfss://{workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{destination_lakehouse_id}/{destination_lakehouse_path}", xml_output, overwrite=True)
    return xml_output

report = run_tests_and_write_output(spark)